In [1]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "7afb4924-43e3-4d91-b3a5-39dbaec4e634"
RESOURCE_GROUP  = "rg-mlab1-zeenatc2c"
WORKSPACE       = "mlw-lab1-zeenatc2c"

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE
)

In [3]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, IdentityConfiguration

ENDPOINT_NAME = "knn-maint-zeenat1"  

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance (notebook model)",
    auth_mode="key",
    identity=IdentityConfiguration(type="system_assigned"),  
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created with Identity:", ENDPOINT_NAME)

Endpoint created with Identity: knn-maint-zeenat1


In [ ]:
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print("Deploying model:", model.name, "version", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_F2s_v2",  # <-- DS2_v2 ki jagah F2s_v2 karein (16 cores free hain)
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

# Send 100% traffic to blue
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

In [4]:
ENDPOINT_NAME = "knn-maint-zeenat1"

try:
    logs = ml_client.online_deployments.get_logs(
        name="blue",
        endpoint_name=ENDPOINT_NAME,
        lines=100
    )
    print(logs)
except Exception as e:
    print("Error retrieving logs:")
    print(e)

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-10-12T11:49:03.439904Z, Message: Start pulling container image
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-10-12T11:49:03.859315Z, Message: Start downloading models
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-10-12T11:49:51.033551Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-10-12T11:49:51.033551Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-10-12T11:49:51.078988Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-10-12T11:49:51.131923Z, Message: Started container inference-server
Kind: Pod, Name: ContainerReady, Type: Normal, Time: 2026-10-12T11:50:15.782649388Z, Message: Container is ready

Container logs:
2026-10-12 13

In [5]:
model = ml_client.models.get(
    name="knn-maintenance-notebook",
    version="1"
)

print("Model name:", model.name)
print("Model version:", model.version)
print("Model path:", model.path)
print("Model type:", model.type)

Model name: knn-maintenance-notebook
Model version: 1
Model path: azureml://subscriptions/7afb4924-43e3-4d91-b3a5-39dbaec4e634/resourceGroups/rg-mlab1-zeenatc2c/workspaces/mlw-lab1-zeenatc2c/datastores/workspaceartifactstore/paths/ExperimentRun/dcid.9c2b89e4-1b21-4400-b378-e27f70b73b9f/model
Model type: mlflow_model


In [6]:
import os

download_dir = "./model_check"

ml_client.models.download(
    name="knn-maintenance-notebook",
    version="1",
    download_path=download_dir
)

print("Downloaded model files:")

for root, dirs, files in os.walk(download_dir):
    for file in files:
        print(os.path.join(root, file))

Downloaded model files:
./model_check/knn-maintenance-notebook/model/conda.yaml
./model_check/knn-maintenance-notebook/model/conda_fixed.yaml
./model_check/knn-maintenance-notebook/model/input_example.json
./model_check/knn-maintenance-notebook/model/MLmodel
./model_check/knn-maintenance-notebook/model/model.pkl
./model_check/knn-maintenance-notebook/model/python_env.yaml
./model_check/knn-maintenance-notebook/model/requirements.txt
./model_check/knn-maintenance-notebook/model/serving_input_payload.json


In [7]:
for root, dirs, files in os.walk(download_dir):
    for file in files:
        if file == "conda.yaml":
            path = os.path.join(root, file)
            print("File:", path)
            with open(path, "r") as f:
                print(f.read())

File: ./model_check/knn-maintenance-notebook/model/conda.yaml
channels:
- conda-forge
dependencies:
- python=3.10.21
- pip<=26.2.1
- pip:
  - mlflow==2.15.1
  - cloudpickle==3.1.2
  - numpy==1.26.4
  - pandas==2.3.3
  - psutil==7.2.2
  - pyarrow==15.0.2
  - scikit-learn==1.7.2
  - scipy==1.15.3
name: mlflow-env



In [8]:
from azure.ai.ml.entities import Environment

import os

model_dir = "./model_check/knn-maintenance-notebook/model"
conda_path = os.path.join(model_dir, "conda_fixed.yaml")

conda_content = """channels:
  - conda-forge
dependencies:
  - python=3.10.21
  - pip
  - pip:
      - mlflow==2.15.1
      - cloudpickle==3.1.2
      - numpy==1.26.4
      - pandas==2.3.3
      - psutil==7.2.2
      - pyarrow==15.0.2
      - scikit-learn==1.7.2
      - scipy==1.15.3
      - setuptools==80.9.0
      - azureml-inference-server-http
      - azureml-ai-monitoring
name: mlflow-env-fixed
"""

with open(conda_path, "w") as f:
    f.write(conda_content)

env = Environment(
    name="knn-maintenance-env",
    description="MLflow KNN environment with compatible setuptools",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    conda_file=conda_path
)

created_env = ml_client.environments.create_or_update(env)

print("Environment registered:", created_env.name)
print("Environment version:", created_env.version)

Environment registered: knn-maintenance-env
Environment version: 7


In [9]:
ENDPOINT_NAME = "knn-maint-zeenat1"

try:
    ml_client.online_deployments.begin_delete(
        name="blue",
        endpoint_name=ENDPOINT_NAME
    ).result()
    print("Old blue deployment deleted.")
except Exception as e:
    print("Delete result:", e)

Old blue deployment deleted.


In [10]:
ENDPOINT_NAME = "knn-maint-zeenat1"

try:
    logs = ml_client.online_deployments.get_logs(
        name="blue",
        endpoint_name=ENDPOINT_NAME,
        lines=100
    )
    print(logs)
except Exception as e:
    print("Could not retrieve logs:")
    print(e)

Could not retrieve logs:
(ResourceNotFound) The Resource 'Microsoft.MachineLearningServices/workspaces/mlw-lab1-zeenatc2c/onlineEndpoints/knn-maint-zeenat1/deployments/blue' under resource group 'rg-mlab1-zeenatc2c' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix
Code: ResourceNotFound
Message: The Resource 'Microsoft.MachineLearningServices/workspaces/mlw-lab1-zeenatc2c/onlineEndpoints/knn-maint-zeenat1/deployments/blue' under resource group 'rg-mlab1-zeenatc2c' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix


In [11]:
from azure.ai.ml.entities import Environment

conda_content = """channels:
  - conda-forge
dependencies:
  - python=3.10.21
  - pip
  - pip:
      - mlflow==2.15.1
      - cloudpickle==3.1.2
      - numpy==1.26.4
      - pandas==2.3.3
      - psutil==7.2.2
      - pyarrow==15.0.2
      - scikit-learn==1.7.2
      - scipy==1.15.3
      - setuptools==80.9.0
      - azureml-inference-server-http
      - azureml-ai-monitoring
      - azureml-contrib-services
name: mlflow-env-fixed
"""

with open("./model_check/knn-maintenance-notebook/model/conda_fixed.yaml", "w") as f:
    f.write(conda_content)

env_v2 = Environment(
    name="knn-maintenance-env",
    description="KNN MLflow environment with Azure ML serving dependencies",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    conda_file="./model_check/knn-maintenance-notebook/model/conda_fixed.yaml"
)

registered_env = ml_client.environments.create_or_update(env_v2)

print("Environment:", registered_env.name)
print("Version:", registered_env.version)

Environment: knn-maintenance-env
Version: 8


In [12]:
ENDPOINT_NAME = "knn-maint-zeenat1"

try:
    ml_client.online_deployments.begin_delete(
        name="blue",
        endpoint_name=ENDPOINT_NAME
    ).result()
    print("Old blue deployment deleted.")
except Exception as e:
    print("Delete result:", e)

Old blue deployment deleted.


In [13]:
logs = ml_client.online_deployments.get_logs(
    name="blue",
    endpoint_name="knn-maint-zeenat1",
    lines=100
)

print(logs)

ResourceNotFoundError: (ResourceNotFound) The Resource 'Microsoft.MachineLearningServices/workspaces/mlw-lab1-zeenatc2c/onlineEndpoints/knn-maint-zeenat1/deployments/blue' under resource group 'rg-mlab1-zeenatc2c' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix
Code: ResourceNotFound
Message: The Resource 'Microsoft.MachineLearningServices/workspaces/mlw-lab1-zeenatc2c/onlineEndpoints/knn-maint-zeenat1/deployments/blue' under resource group 'rg-mlab1-zeenatc2c' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix

In [14]:
import os

os.makedirs("./deployment", exist_ok=True)

score_code = r'''
import json
import os
import pandas as pd
import mlflow.pyfunc

def init():
    global model
    model_dir = os.environ.get("AZUREML_MODEL_DIR")
    model = mlflow.pyfunc.load_model(model_dir)

def run(raw_data):
    try:
        data = json.loads(raw_data)
        columns = data["input_data"]["columns"]
        rows = data["input_data"]["data"]

        df = pd.DataFrame(rows, columns=columns)
        predictions = model.predict(df)

        return {"predictions": predictions.tolist()}
    except Exception as e:
        return {"error": str(e)}
'''

with open("./deployment/score.py", "w") as f:
    f.write(score_code)

print("Scoring script created:", os.path.exists("./deployment/score.py"))

Scoring script created: True


In [15]:
score_code = r'''
import os
import json
import pandas as pd
import mlflow.pyfunc

def init():
    global model
    model_dir = os.path.join(
        os.environ["AZUREML_MODEL_DIR"], "model"
    )
    model = mlflow.pyfunc.load_model(model_dir)

def run(raw_data):
    data = json.loads(raw_data)
    input_data = data["input_data"]
    df = pd.DataFrame(
        input_data["data"],
        columns=input_data["columns"]
    )
    predictions = model.predict(df)
    return {"predictions": predictions.tolist()}
'''

with open("./deployment/score.py", "w") as f:
    f.write(score_code)

print("score.py updated")

score.py updated


In [16]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="knn-maint-zeenat1",
    model=ml_client.models.get(
        name="knn-maintenance-notebook", version="1"
    ),
    environment=ml_client.environments.get(
        name="knn-maintenance-env", version="2"
    ),
    code_configuration=CodeConfiguration(
        code="./deployment",
        scoring_script="score.py"
    ),
    instance_type="Standard_F2s_v2",
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(
    deployment
).result()

print("Deployment operation completed.")

Check: endpoint knn-maint-zeenat1 exists
Uploading deployment (0.0 MBs): 100%|██████████| 1372/1372 [00:00<00:00, 17261.18it/s]




.............................................................................................Deployment operation completed.


In [18]:
deployment_status = ml_client.online_deployments.get(
    name="blue",
    endpoint_name="knn-maint-zeenat1"
)

print("Provisioning state:", deployment_status.provisioning_state)
print("Instance count:", deployment_status.instance_count)

Provisioning state: DeploymentProvisioningState.SUCCEEDED
Instance count: 1


In [19]:
endpoint = ml_client.online_endpoints.get(
    name="knn-maint-zeenat1"
)

endpoint.traffic = {"blue": 100}

ml_client.online_endpoints.begin_create_or_update(
    endpoint
).result()

print("100% traffic assigned to blue.")

100% traffic assigned to blue.


In [20]:
import json

sample_data = {
    "input_data": {
        "columns": [
            "air_temp_k",
            "process_temp_k",
            "rpm",
            "torque_nm",
            "tool_wear_min",
            "type"
        ],
        "data": [
            [298.1, 308.6, 1551.0, 42.8, 0.0, "M"],
            [302.5, 310.9, 1325.0, 68.2, 215.0, "L"]
        ]
    }
}

with open("sample_request.json", "w") as f:
    json.dump(sample_data, f)

print("Request file created.")

Request file created.


In [21]:
result = ml_client.online_endpoints.invoke(
    endpoint_name="knn-maint-zeenat1",
    request_file="sample_request.json",
    deployment_name="blue"
)

print("Prediction response:")
print(result)

Prediction response:
{"predictions": [0, 1]}


In [22]:
import os

endpoint = ml_client.online_endpoints.get(
    name="knn-maint-zeenat1"
)
keys = ml_client.online_endpoints.get_keys(
    name="knn-maint-zeenat1"
)

os.environ["AML_ENDPOINT_URL"] = endpoint.scoring_uri
os.environ["AML_ENDPOINT_KEY"] = keys.primary_key

print("Endpoint URL and key configured for REST test.")

Endpoint URL and key configured for REST test.


In [23]:
import os

endpoint = ml_client.online_endpoints.get(
    name="knn-maint-zeenat1"
)
keys = ml_client.online_endpoints.get_keys(
    name="knn-maint-zeenat1"
)

print("Endpoint URL:", endpoint.scoring_uri)
print("Key available:", bool(keys.primary_key))

Endpoint URL: https://knn-maint-zeenat1.koreacentral.inference.ml.azure.com/score
Key available: True


In [24]:
endpoint = ml_client.online_endpoints.get(
    name="knn-maint-zeenat1"
)

keys = ml_client.online_endpoints.get_keys(
    name="knn-maint-zeenat1"
)

print("URL:", endpoint.scoring_uri)
print("Primary key available:", bool(keys.primary_key))


URL: https://knn-maint-zeenat1.koreacentral.inference.ml.azure.com/score
Primary key available: True


In [25]:
keys = ml_client.online_endpoints.get_keys(
    name="knn-maint-zeenat1"
)

print("Primary key available:", bool(keys.primary_key))


Primary key available: True


In [26]:
import os
import getpass

keys = ml_client.online_endpoints.get_keys(
    name="knn-maint-zeenat1"
)

os.environ["AML_ENDPOINT_URL"] = (
    "https://knn-maint-zeenat1.koreacentral.inference.ml.azure.com/score"
)

os.environ["AML_ENDPOINT_KEY"] = keys.primary_key

print("Endpoint URL configured.")
print("Key configured:", bool(os.environ["AML_ENDPOINT_KEY"]))

Endpoint URL configured.
Key configured: True


In [27]:
import requests

payload = {
    "input_data": {
        "columns": [
            "air_temp_k", "process_temp_k", "rpm",
            "torque_nm", "tool_wear_min", "type"
        ],
        "data": [
            [298.1, 308.6, 1551.0, 42.8, 0.0, "M"],
            [302.5, 310.9, 1325.0, 68.2, 215.0, "L"]
        ]
    }
}

response = requests.post(
    os.environ["AML_ENDPOINT_URL"],
    headers={
        "Authorization": f"Bearer {os.environ['AML_ENDPOINT_KEY']}",
        "Content-Type": "application/json"
    },
    json=payload,
    timeout=60
)

print("HTTP Status:", response.status_code)
print("Response:", response.text)

HTTP Status: 200
Response: {"predictions": [0, 1]}
